# Proyecto 3 – Simulación de Carrera F1 en CUDA


In [ ]:
# Instalación de extensión nvcc4jupyter (si procede)
!pip -q install nvcc4jupyter
%load_ext nvcc4jupyter

In [ ]:
# Detección de GPU y arquitectura
import subprocess, re
info = subprocess.getoutput("nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader,nounits")
if 'not found' in info.lower() or not info.strip():
    print('No se encontró GPU NVIDIA; se mostrarán kernels pero no se ejecutarán.')
    gpu_name='N/A'; gpu_arch='sm_70'
else:
    parts=[p.strip() for p in info.split(',')]
    gpu_name=parts[0]; compute=parts[1].replace('.','')
    gpu_arch=f'sm_{compute}'
print('GPU:', gpu_name)
print('Arquitectura nvcc:', gpu_arch)

In [ ]:
%%cuda
// =============================================================
// Simulación F1 CUDA (Benchmark + Race por variante)
// Variantes: GlobalOnly, ConstGlobal, ConstSharedGlobal
// Emite: RESULT (rendimiento) y PILOT_SUMMARY (ranking de carrera)
// =============================================================
#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>
#include <algorithm>
#ifndef CHECK
#define CHECK(x) do { cudaError_t e=(x); if(e!=cudaSuccess){\
    printf("CUDA Error %s (%d)\n", cudaGetErrorString(e), e); exit(1);} } while(0)
#endif
// Parámetros carrera
static const int N = 20; // pilotos
static const int V = 30; // vueltas
static const int S = 4;  // sectores
static const int reps = 10; // repeticiones benchmark
// Roster pilotos + equipos
const char* DRIVERS[N] = {
    "Max Verstappen","Liam Lawson","Charles Leclerc","Lewis Hamilton","George Russell","Andrea Kimi Antonelli",
    "Lando Norris","Oscar Piastri","Fernando Alonso","Lance Stroll","Pierre Gasly","Jack Doohan",
    "Yuki Tsunoda","Isack Hadjar","Nico Hulkenberg","Gabriel Bortoleto","Carlos Sainz","Alexander Albon",
    "Oliver Bearman","Esteban Ocon"
};
const char* TEAMS[N] = {
    "Red Bull","Red Bull","Ferrari","Ferrari","Mercedes","Mercedes","McLaren","McLaren","Aston Martin","Aston Martin",
    "Alpine","Alpine","Visa Cash App RB","Visa Cash App RB","Kick Sauber","Kick Sauber","Williams","Williams","Haas F1","Haas F1"
};
// Estructuras resultado por variante
struct ResG { int best; int total; };
struct ResC { int best; int total; };
struct ResS { int best; int total; };
// ---------------- Kernel GlobalOnly ----------------
__global__ void kGlobal(int *times, ResG *res, int *gBest, int V, int S, int pitPenalty, float pitProb){
    int stride = blockDim.x * gridDim.x;
    for(int idx = blockIdx.x*blockDim.x + threadIdx.x; idx < N*V; idx += stride){
        int p = idx / V; int lap = idx % V; unsigned int seed = 2222u + idx*17u;
        int lapTime = 0;
        for(int sct=0;sct<S;++sct){ seed=seed*1103515245u+12345u; int var=(seed>>16)%7; lapTime += 25 + var; }
        seed=seed*1103515245u+12345u; float r = ((seed>>8)&0xFFFF)/65535.0f; if(r<pitProb) lapTime += pitPenalty;
        atomicAdd(&res[p].total, lapTime); atomicMin(&res[p].best, lapTime); atomicMin(gBest, lapTime);
    }
}
// ---------------- Kernel ConstGlobal ----------------
__constant__ int cBase[8]; __constant__ int cPen[3]; __constant__ float cProb;
__global__ void kConst(int *times, ResC *res, int *gBest, int V, int S){
    int stride = blockDim.x * gridDim.x;
    for(int idx = blockIdx.x*blockDim.x + threadIdx.x; idx < N*V; idx += stride){
        int p = idx / V; int lap = idx % V; unsigned int seed = 3333u + idx*23u; int lapTime=0;
        for(int sct=0;sct<S;++sct){ seed=seed*1103515245u+12345u; int var=(seed>>16)%7; lapTime += cBase[sct] + var; }
        int stage = (lap < V/3)?0:(lap < 2*V/3?1:2); seed=seed*1103515245u+12345u; float r=((seed>>8)&0xFFFF)/65535.0f; if(r<cProb) lapTime += cPen[stage];
        atomicAdd(&res[p].total, lapTime); atomicMin(&res[p].best, lapTime); atomicMin(gBest, lapTime);
    }
}
// ---------------- Kernel ConstSharedGlobal ----------------
__constant__ int cBase3[8]; __constant__ int cPen3[3]; __constant__ float cProb3;
__global__ void kShared(int *times, ResS *res, int *gBest, int V, int S){
    extern __shared__ int sh[]; int *base = sh; int *partial = sh + S;
    if(threadIdx.x < S) base[threadIdx.x] = cBase3[threadIdx.x];
    __syncthreads();
    int stride = blockDim.x * gridDim.x; int localBest = 1e9;
    for(int idx = blockIdx.x*blockDim.x + threadIdx.x; idx < N*V; idx += stride){
        int p = idx / V; int lap = idx % V; unsigned int seed = 4444u + idx*31u + threadIdx.x; int lapTime=0;
        for(int sct=0;sct<S;++sct){ seed=seed*1103515245u+12345u; int var=(seed>>16)%7; lapTime += base[sct] + var; }
        int stage=(lap<V/3)?0:(lap<2*V/3?1:2); seed=seed*1103515245u+12345u; float r=((seed>>8)&0xFFFF)/65535.0f; if(r<cProb3) lapTime += cPen3[stage];
        atomicAdd(&res[p].total, lapTime); atomicMin(&res[p].best, lapTime); if(lapTime<localBest) localBest = lapTime;
    }
    partial[threadIdx.x] = localBest; __syncthreads();
    for(int off = blockDim.x/2; off>0; off >>=1){ if(threadIdx.x < off){ partial[threadIdx.x] = min(partial[threadIdx.x], partial[threadIdx.x+off]); } __syncthreads(); }
    if(threadIdx.x==0) atomicMin(gBest, partial[0]);
}
// ---------------- Utilidad: imprimir ranking ----------------
template<typename T>
void printRanking(const char* tag, T* hostRes){
    int order[N]; for(int i=0;i<N;++i) order[i]=i;
    std::sort(order, order+N, [&](int a,int b){ return hostRes[a].total < hostRes[b].total; });
    for(int pos=0; pos<N; ++pos){ int p = order[pos];
        printf("PILOT_SUMMARY,%s,%d,%d,%s,%s,%d,%d\n", tag, pos+1, p, DRIVERS[p], TEAMS[p], hostRes[p].best, hostRes[p].total);
    }
}
// ---------------- Rutina benchmark+race genérica ----------------
template<typename T, typename Kernel, typename Setup>
void runVariant(const char* tag, Kernel kernel, Setup setup, bool useShared){
    size_t count = (size_t)N*V*S; int *d_times; T *d_res; int *d_best;
    CHECK(cudaMalloc(&d_times, count*sizeof(int))); CHECK(cudaMalloc(&d_res, N*sizeof(T))); CHECK(cudaMalloc(&d_best, sizeof(int)));
    dim3 block(256); dim3 grid((N*V + block.x -1)/block.x); if(grid.x>1024) grid.x=1024;
    cudaEvent_t start, stop; CHECK(cudaEventCreate(&start)); CHECK(cudaEventCreate(&stop));
    for(int run=0; run<reps; ++run){ CHECK(cudaMemset(d_times,0,count*sizeof(int))); T host[N]; for(int i=0;i<N;++i){ host[i].best=1e9; host[i].total=0; } int init=1e9;
        CHECK(cudaMemcpy(d_res, host, N*sizeof(T), cudaMemcpyHostToDevice)); CHECK(cudaMemcpy(d_best,&init,sizeof(int),cudaMemcpyHostToDevice)); setup(); CHECK(cudaEventRecord(start));
        if(useShared){ size_t shBytes = (S + block.x)*sizeof(int); kernel<<<grid,block,shBytes>>>(d_times,d_res,d_best,V,S); }
        else { kernel<<<grid,block>>>(d_times,d_res,d_best,V,S); }
        CHECK(cudaGetLastError()); CHECK(cudaEventRecord(stop)); CHECK(cudaEventSynchronize(stop)); float ms; CHECK(cudaEventElapsedTime(&ms,start,stop)); int gBest;
        CHECK(cudaMemcpy(&gBest,d_best,sizeof(int),cudaMemcpyDeviceToHost)); printf("RESULT,%s,%d,%.3f,%d\n", tag, run+1, ms, gBest);
    }
    // Race final
    CHECK(cudaMemset(d_times,0,count*sizeof(int))); T hostRace[N]; for(int i=0;i<N;++i){ hostRace[i].best=1e9; hostRace[i].total=0; } int initRace=1e9;
    CHECK(cudaMemcpy(d_res, hostRace, N*sizeof(T), cudaMemcpyHostToDevice)); CHECK(cudaMemcpy(d_best,&initRace,sizeof(int),cudaMemcpyHostToDevice)); setup();
    if(useShared){ size_t shBytes=(S + block.x)*sizeof(int); kernel<<<grid,block,shBytes>>>(d_times,d_res,d_best,V,S); }
    else { kernel<<<grid,block>>>(d_times,d_res,d_best,V,S); }
    CHECK(cudaDeviceSynchronize()); CHECK(cudaMemcpy(hostRace,d_res,N*sizeof(T),cudaMemcpyDeviceToHost)); printRanking(tag, hostRace);
    CHECK(cudaFree(d_times)); CHECK(cudaFree(d_res)); CHECK(cudaFree(d_best)); cudaEventDestroy(start); cudaEventDestroy(stop);
}
// ---------------- Main ----------------
int main(){
    // GlobalOnly
    runVariant<ResG>("GlobalOnly",
        [] __device__ (int* t, ResG* r, int* gB, int Vv, int Ss){ kGlobal(t,r,gB,Vv,Ss,15,0.05f); },
        [](){ /* no setup */ }, false);
    // ConstGlobal setup
    int hBase[8]; for(int i=0;i<S;++i) hBase[i]=25+(i%2); int hPen[3]={12,15,10}; float hProb=0.06f;
    runVariant<ResC>("ConstGlobal",
        [] __device__ (int* t, ResC* r, int* gB, int Vv, int Ss){ kConst(t,r,gB,Vv,Ss); },
        [&](){ CHECK(cudaMemcpyToSymbol(cBase,hBase,8*sizeof(int))); CHECK(cudaMemcpyToSymbol(cPen,hPen,3*sizeof(int))); CHECK(cudaMemcpyToSymbol(cProb,&hProb,sizeof(float))); }, false);
    // ConstSharedGlobal setup
    int hBase3[8]; for(int i=0;i<S;++i) hBase3[i]=25+(i%3); int hPen3[3]={10,14,11}; float hProb3=0.055f;
    runVariant<ResS>("ConstSharedGlobal",
        [] __device__ (int* t, ResS* r, int* gB, int Vv, int Ss){ kShared(t,r,gB,Vv,Ss); },
        [&](){ CHECK(cudaMemcpyToSymbol(cBase3,hBase3,8*sizeof(int))); CHECK(cudaMemcpyToSymbol(cPen3,hPen3,3*sizeof(int))); CHECK(cudaMemcpyToSymbol(cProb3,&hProb3,sizeof(float))); }, true);
    return 0;
}


In [ ]:
# Parser organizado: lee últimos outputs en lugar del código fuente
import re, pandas as pd
from IPython import get_ipython
# Recorremos los outputs almacenados (_oh) buscando líneas RESULT y PILOT_SUMMARY
all_text = []
for k,v in get_ipython().user_ns.get('_oh', {}).items():
    if isinstance(v,str):
        all_text.append(v)
text = '\n'.join(all_text)
pat_result = re.compile(r"RESULT,(\w+),(\d+),(\d+\.\d+),(\d+)")
pat_pilot  = re.compile(r"PILOT_SUMMARY,(\w+),(\d+),(\d+),([^,]+),([^,]+),(\d+),(\d+)")
bench_rows=[]; pilot_rows=[]
for line in text.splitlines():
    if (m:=pat_result.match(line.strip())):
        variant, run, ms, best = m.groups(); bench_rows.append({"variant":variant,"run":int(run),"ms":float(ms),"best_lap":int(best)})
    elif (m:=pat_pilot.match(line.strip())):
        variant, pos, idx, name, team, bestLap, total = m.groups()
        pilot_rows.append({"variant":variant,"position":int(pos),"pilot_index":int(idx),"driver":name,"team":team,"best_lap":int(bestLap),"total_time":int(total)})
if not bench_rows:
    print("No hay datos RESULT. Ejecuta la celda de simulación primero.")
else:
    df_bench = pd.DataFrame(bench_rows)
    summary = df_bench.groupby('variant').agg(avg_ms=('ms','mean'), std_ms=('ms','std'), best_global=('best_lap','min'))
    display(summary.sort_values('avg_ms'))
if pilot_rows:
    df_pilot = pd.DataFrame(pilot_rows)
    ranking = df_pilot.sort_values(['variant','position'])
    print("Top 5 por variante:")
    display(ranking.groupby('variant').head(5))
    df_pilot.to_csv('race_pilots.csv', index=False)
else:
    print("No se detectaron PILOT_SUMMARY aún.")


In [ ]:
# Visualizaciones "cool": barras, boxplot y gauge pseudo
import pandas as pd, plotly.express as px, math
try:
    df = pd.read_csv('bench_raw.csv')
except FileNotFoundError:
    print('Primero ejecuta las celdas de medición y parsing.')
else:
    # Barras promedio
    avg = df.groupby('variant')['ms'].mean().reset_index().sort_values('ms')
    fig1 = px.bar(avg, x='variant', y='ms', title='Tiempo promedio por variante (ms)', color='variant', text='ms')
    fig1.update_layout(template='plotly_dark')
    fig1.show()
    # Boxplot distribuciones
    fig2 = px.box(df, x='variant', y='ms', title='Distribución de tiempos (ms)', color='variant')
    fig2.update_layout(template='plotly_dark')
    fig2.show()
    # Gauge simple (usando scatter polar) mejor vuelta global más rápida
    best_global = df['best_lap'].min()
    span = df['best_lap'].max() - best_global
    norm = 0 if span == 0 else (best_global - df['best_lap'].min())/span
    import plotly.graph_objects as go
    fig3 = go.Figure(go.Indicator(mode="gauge+number", value=best_global, title={'text':'Mejor Vuelta Global (seg)'}, gauge={'axis':{'range':[best_global, best_global+span or 1]}, 'bar':{'color':'lime'}}))
    fig3.update_layout(template='plotly_dark')
    fig3.show()
    # Tabla estilizada
    from IPython.display import display
    pivot = df.groupby('variant').agg(avg_ms=('ms','mean'), min_best=('best_lap','min'))
    display(pivot.style.background_gradient(cmap='viridis').format({'avg_ms':'{:.2f}','min_best':'{}'}))


In [ ]:
# Animación textual simple del avance de 5 pilotos (mock con DataFrame)
import pandas as pd, time, random
try:
    df = pd.read_csv('bench_raw.csv')
except FileNotFoundError:
    print('Ejecuta mediciones primero.')
else:
    # Generar tiempos sintéticos de 5 pilotos en 10 vueltas basados en mejor vuelta global
    best = df['best_lap'].min()
    pilotos = ['Verstappen','Hamilton','Norris','Leclerc','Sainz']
    vueltas = 10
    for lap in range(1, vueltas+1):
        line = []
        for p in pilotos:
            base = best + 5 + random.randint(0,8)
            if random.random() < 0.05: # pit
                base += 12
                ptxt = f"{p}:{base}*"
            else:
                ptxt = f"{p}:{base}"
            line.append(ptxt)
        print(f"Vuelta {lap:02d} | " + " | ".join(line))
        time.sleep(0.15)
    print("(*) incluye penalización pit stop simulada")
